In [1]:
!pip install -q -U langchain langchain-core langchain-community langchain-google-genai langchain-chroma langchain-classic pypdf docx2txt sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18

In [2]:
import os
os.environ["GOOGLE_API_KEY"]=""

In [3]:
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ['LANGCHAIN_API_KEY']=""
os.environ['LANGCHAIN_PROJECT']='rag-chatbot'

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_response=llm.invoke("Tell me a simple joke about coding")
llm_response

AIMessage(content='Why do programmers prefer dark mode?\n\nBecause bugs are harder to see in the dark!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dc946-90fd-7752-bca7-1f86b2b33e07-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 1921, 'total_tokens': 1929, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 1903}})

In [5]:
from langchain_core.output_parsers import StrOutputParser

output_parser=StrOutputParser()
clean_text = output_parser.invoke(llm_response)
clean_text

'Why do programmers prefer dark mode?\n\nBecause bugs are harder to see in the dark!'

In [6]:
chain=llm | output_parser
chain.invoke("Tell me a story!")

"Once, in a valley cradled by ancient, mist-kissed mountains, stood a forest known as the Whispering Woods. It wasn't just a name; the trees truly whispered. The rustle of leaves sounded like hushed secrets, the wind sighing through the pines carried forgotten lullabies, and the babbling brook sang a constant, gentle tune. Every creature, from the smallest beetle to the soaring hawk, contributed to this symphony of soft sounds.\n\nElara lived in the village at the edge of the woods. She was a quiet girl, with eyes the color of moss after rain and an uncanny ability to hear the faintest sounds. While others found the constant murmuring of the woods a pleasant background, Elara understood its language. She knew when the oaks were sharing tales of ancient storms, when the birches gossiped about the passing deer, and when the willows wept for lost sunlight.\n\nOne crisp autumn morning, Elara awoke to an unsettling silence. It wasn't the silence of deep snow, which had its own soft blanket 

In [7]:
from typing import List
from pydantic import BaseModel, Field

class ReviewDSA(BaseModel):
  topic: str=Field(description="Name of the DSA topic")
  summary: str=Field(description="Briedf summary of the review")
  rating: float=Field(description="Overall rating out of 5")
  pros: List[str]=Field(description="List of positive aspects")

prompt_text='''
 Just started learning Dynamic Programming, and wow, this topic is mind-blowing! The way it optimizes
    recursive solutions is just amazing. It really helps in solving problems efficiently that involve
    overlapping subproblems, like Fibonacci and Knapsack.

    But not gonna lie, it's tough at first. You really need to practice a lot to get the hang of thinking
    in terms of subproblems. Also, memorization techniques can be confusing, and it’s easy to mess up
    the transition from recursion to tabulation.

    Overall, I'd rate it a 4.5 out of 5. Once you get it, it’s a game-changer for problem-solving.
    Definitely worth the effort!
'''
structured_llm=llm.with_structured_output(ReviewDSA)
output=structured_llm.invoke(prompt_text)
print(output)
print(output.pros)

topic='Dynamic Programming' summary="The user finds Dynamic Programming mind-blowing for optimizing recursive solutions, especially for problems with overlapping subproblems like Fibonacci and Knapsack. They acknowledge it's tough initially due to the need for practice, understanding memorization, and transitioning to tabulation, but ultimately consider it a game-changer and worth the effort." rating=4.5 pros=['Optimizes recursive solutions', 'Helps in solving problems efficiently that involve overlapping subproblems', 'Game-changer for problem-solving', 'Worth the effort']
['Optimizes recursive solutions', 'Helps in solving problems efficiently that involve overlapping subproblems', 'Game-changer for problem-solving', 'Worth the effort']


In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt.invoke({"topic" : "sports"})


ChatPromptValue(messages=[HumanMessage(content='Tell me a joke about sports', additional_kwargs={}, response_metadata={})])

In [9]:
chain=prompt | llm|output_parser
result=chain.invoke({"topic" : "coding"})
print(result)

Why do programmers always mix up Christmas and Halloween?

Because Oct 31 == Dec 25.


In [10]:
from langchain_core.messages import HumanMessage,SystemMessage
messages=[
    SystemMessage(content="You are a helpful coding assistant!"),
    HumanMessage(content="Tell me a joke about Debugging in code")
]
response=llm.invoke(messages)
print(response)

content='Why did the programmer break up with the debugger?\n\nBecause it kept pointing out all his flaws!' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019dc947-2149-77a0-8836-77ee975fa561-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 17, 'output_tokens': 2327, 'total_tokens': 2344, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 2307}}


In [11]:
template=ChatPromptTemplate([
    ("system","You are a helpful coding assistant"),
    ("human","tell me a joke about {topic}")
])
chain=template | llm
result=chain.invoke({"topic":"coding"})
print(result)

content='Why do programmers prefer dark mode?\n\nBecause bugs are attracted to light!' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019dc947-531b-7252-9b55-408451f15658-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 13, 'output_tokens': 352, 'total_tokens': 365, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 337}}


In [12]:
!pip install Docx2txt

In [13]:
from langchain_community.document_loaders import PyPDFLoader,Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.documents import Document
import os

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200,length_function=len)
docx_loader=Docx2txtLoader("/content/docs/DBMS_Full_Notes_Study_Notes.docx")
documents=docx_loader.load()

splits=text_splitter.split_documents(documents)
print(f"splitted with {len(splits)} chunks")
print(len(documents))


splitted with 17 chunks
1


In [14]:
documents[0]

Document(metadata={'source': '/content/docs/DBMS_Full_Notes_Study_Notes.docx'}, page_content="DBMS Full Notes\n\nSimplified revision notes from the uploaded PDF\n\nMade in an easy exam-revision style: definitions, intuition, comparisons, and quick memory hooks.\n\n\n\n\n\n1. Introduction to DBMS\n\nThink of data as raw material and information as processed meaning. A DBMS sits in between storage and the user so data can be stored, searched, updated, and protected in a controlled way.\n\nData vs Information\n\nData\n\nInformation\n\nRaw facts, numbers, symbols, and observations.\n\nMeaningful output after processing data.\n\nUnorganized by itself.\n\nOrganized and useful for decisions.\n\nCan be stored but may not explain anything.\n\nGives context and helps in decision-making.\n\n\n\nKey terms\n\nDatabase: an electronic place where data is stored so it can be accessed and updated easily.\n\nDBMS: software that manages the database and the operations on it.\n\nMain job of DBMS: store da

In [15]:
splits[0]

Document(metadata={'source': '/content/docs/DBMS_Full_Notes_Study_Notes.docx'}, page_content='DBMS Full Notes\n\nSimplified revision notes from the uploaded PDF\n\nMade in an easy exam-revision style: definitions, intuition, comparisons, and quick memory hooks.\n\n\n\n\n\n1. Introduction to DBMS\n\nThink of data as raw material and information as processed meaning. A DBMS sits in between storage and the user so data can be stored, searched, updated, and protected in a controlled way.\n\nData vs Information\n\nData\n\nInformation\n\nRaw facts, numbers, symbols, and observations.\n\nMeaningful output after processing data.\n\nUnorganized by itself.\n\nOrganized and useful for decisions.\n\nCan be stored but may not explain anything.\n\nGives context and helps in decision-making.\n\n\n\nKey terms\n\nDatabase: an electronic place where data is stored so it can be accessed and updated easily.\n\nDBMS: software that manages the database and the operations on it.\n\nMain job of DBMS: store da

In [16]:
def load_documents(folder_path : str)-> List[Document]:
  documents=[]
  for filename in os.listdir(folder_path):
    file_path=os.path.join(folder_path,filename)
    if filename.endswith('.pdf'):
      loader=PyPDFLoader(file_path)
    elif filename.endswith('.docx'):
      loader=Docx2txtLoader(file_path)
    else:
      print(f"unsupported file type : {filename}")
      continue
    documents.extend(loader.load())
  return documents

folder_path='/content/docs'
documents=load_documents(folder_path)
print(f"loaded {len(documents)} from folder")

splits=text_splitter.split_documents(documents)
print(f"splitted documents into {len(splits)} chunks")


loaded 3 from folder
splitted documents into 21 chunks


In [17]:
documents[1]

Document(metadata={'producer': 'Skia/PDF m138', 'creator': 'Chromium', 'creationdate': '2026-02-21T16:06:48+00:00', 'title': 'vite-project', 'moddate': '2026-02-21T16:06:48+00:00', 'source': '/content/docs/Time-Complexity-Based-on-Constraints.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='You Asked\nhow to know which till how much time complexity the problem is \ngoing to accept by looking at the constraints\nGemini 3 Pro\nFiguring out the acceptable time complexity before writing a single line of code is one of the most\npowerful skills you can develop for algorithmic problem-solving. It saves you from spending an\nhour on a beautiful solution only to hit a "Time Limit Exceeded" (TLE) error.\nThe secret lies in the  Rule.\nMost online judges (like LeetCode, Codeforces, etc.) give you about 1 to 2 seconds of execution\ntime per test case. A standard modern server can process roughly  basic operations per second\nin compiled languages like C++.\nTo figure out what 

In [18]:
splits[4]

Document(metadata={'source': '/content/docs/DBMS_Full_Notes_Study_Notes.docx'}, page_content='Binary: two entity sets participate.\n\nTernary: three entity sets participate.\n\nCardinality: one-to-one, one-to-many, many-to-one, or many-to-many.\n\nParticipation: partial or total.\n\n4. Extended ER Features\n\nSpecialisation: top-down process. One entity set is split into smaller sub-entity sets based on differences.\n\nGeneralisation: bottom-up process. Similar entity sets are merged into a higher-level superclass.\n\nAttribute inheritance: subclasses inherit attributes of the superclass.\n\nParticipation inheritance: if a parent participates in a relationship, child entities also participate.\n\nAggregation: treat a relationship as a higher-level entity so we can model relationship among relationships.\n\n5. Relational Model\n\nThe relational model stores data in tables. Each table is a relation, each row is a tuple, and each column is an attribute.\n\nBasic terms\n\nRelation = table\

In [19]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

document_embedding = embedding.embed_documents([split.page_content for split in splits])
print(f"created embedding for {len(document_embedding)} documents chunks")

created embedding for 21 documents chunks


In [20]:
from langchain_chroma import Chroma

embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
collection_name="my_collection"
vectorstore=Chroma.from_documents(
    collection_name=collection_name,
    documents=splits,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

print("vector store created and persisted to './chroma_db' ")


vector store created and persisted to './chroma_db' 


In [21]:
query="what is the summary?"

search_results=vectorstore.similarity_search(query,k=2)
print(f"\n Top 2 most relevant search for the query : '{query}' \n ")
for i,result in enumerate(search_results,1):
  print(f"result {i}:")
  print(f"source: {result.metadata.get('source','Unknown')}")
  print(f"content : {result.page_content}")
  print()


 Top 2 most relevant search for the query : 'what is the summary?' 
 
result 1:
source: /content/docs/DBMS_Full_Notes_Study_Notes.docx
content : 9. Normalisation

Normalization is the cleanup step of database design. The goal is simple: reduce duplication and remove anomalies.

Functional dependency

X -> Y means Y depends on X.

Determinant: left side of FD.

Dependent: right side of FD.

Armstrong's axioms: reflexive, augmentation, transitivity.

Why normalize?

To reduce redundancy.

To avoid insertion anomaly.

To avoid deletion anomaly.

To avoid update anomaly and inconsistency.

Normal forms

1NF: each cell must be atomic; no multivalued attributes.

2NF: in 1NF and no partial dependency; every non-prime attribute depends on the whole primary key.

3NF: in 2NF and no transitive dependency; non-prime attributes should not depend on other non-prime attributes.

BCNF: for every FD A -> B, A must be a super key.

10. Transactions

A transaction is a logical unit of work. Either eve

In [22]:
retriever=vectorstore.as_retriever(search_kwargs={"k":2})
retriever_results=retriever.invoke("What is a Database?")
print(retriever_results)

[Document(id='352dd397-7f70-40c9-964a-622f59982df6', metadata={'source': '/content/docs/DBMS_Full_Notes_Study_Notes.docx'}, page_content='DBMS Full Notes\n\nSimplified revision notes from the uploaded PDF\n\nMade in an easy exam-revision style: definitions, intuition, comparisons, and quick memory hooks.\n\n\n\n\n\n1. Introduction to DBMS\n\nThink of data as raw material and information as processed meaning. A DBMS sits in between storage and the user so data can be stored, searched, updated, and protected in a controlled way.\n\nData vs Information\n\nData\n\nInformation\n\nRaw facts, numbers, symbols, and observations.\n\nMeaningful output after processing data.\n\nUnorganized by itself.\n\nOrganized and useful for decisions.\n\nCan be stored but may not explain anything.\n\nGives context and helps in decision-making.\n\n\n\nKey terms\n\nDatabase: an electronic place where data is stored so it can be accessed and updated easily.\n\nDBMS: software that manages the database and the ope

In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template="""
Answer the question based only on the following context:
{context}
Question:{question}
Answer: """


prompt=ChatPromptTemplate.from_template(template)

def docs2str(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain=(
    {"context":retriever | docs2str,"question":RunnablePassthrough()}
    | prompt
    | llm
    |StrOutputParser()
)

In [24]:
question="What is the time complexity of backtracking?"
response=rag_chain.invoke(question)
print(f"question: {question}")
print(f"response: {response}")

question: What is the time complexity of backtracking?
response: The time complexity of backtracking is O(N!) or O(N^6) for N <= 12.


In [25]:
from decimal import Context
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.chains import create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

context_q_system_prompt="""
Given a chat history and latest user question
 which might refernce context in chat history,
formulate a standalone question which can be understood without the chat history.
Do not answer the question,
just formulate it if needed and otherwise return as it is.
"""

context_q_prompt = ChatPromptTemplate.from_messages([
    ("system", context_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

context_chain = context_q_prompt | llm | StrOutputParser()
print(context_chain.invoke({"input":"Compare the time complexity of linear scan and binary search. ","chat_history":[]}))

Compare the time complexity of linear scan and binary search.


In [26]:
from langchain_classic.chains import create_retrieval_chain

history_aware_retriever=create_history_aware_retriever(
    llm,retriever,context_q_prompt
)

qa_prompt=ChatPromptTemplate.from_messages([
     ("system","you are a helpfull AI assistant. use following context to answer the user's question."),
     ("system","Context:{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human","{input}"),
])

question_answer_chain=create_stuff_documents_chain(llm,qa_prompt)
rag_chain=create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [27]:
from langchain_core.messages import HumanMessage,AIMessage

chat_history=[]
question1="Time complexity of linear search?"
answer1=rag_chain.invoke({"input":question1,"chat_history":chat_history})['answer']
chat_history.extend([
    HumanMessage(content=question1),
    AIMessage(content=answer1)
])

print(question1)
print(answer1)

Time complexity of linear search?
Based on the provided context, the time complexity of a linear scan (which includes linear search) is **O(N)**.

This is indicated in the "Constraint Cheat Sheet" where `O(N)` is listed as the maximum acceptable complexity for algorithms like "Linear scans."
